<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN-NEW/blob/main/07_rag/information_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Information Retrieval: TF-IDF, BM25, and Dense Search

This notebook builds the "R" in RAG: the retrieval step that finds the right documents before any model answers from them. We build three retrieval engines on one corpus (TF-IDF, BM25, and dense embeddings) and watch them disagree, since the keyword engines retrieve on shared words while the dense engine retrieves on meaning. We finish on the corpus the course is built on, the Helcon help center, running the semantic search that the next session turns into the Helcon Support Copilot.


In [ ]:
# Setup, run this cell first.
# It installs the libraries, the NLTK assets, and pre-downloads the two embedding
# models of the session (~600 MB), so every later cell loads from cache instantly.
!pip install -q scikit-learn rank_bm25 sentence-transformers nltk numpy

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, quiet=True)

from huggingface_hub import snapshot_download
for repo in ["sentence-transformers/all-MiniLM-L6-v2",
             "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"]:
    snapshot_download(repo)
    print(f"cached: {repo}")

# On Colab we also need the course repository, it carries the Helcon corpus and the shared SDK.
import os
if not os.path.exists("NLP-MBDS-EN") and not os.path.exists("../system/system_io.py"):
    !git clone -q https://github.com/acastellanos-ie/NLP-MBDS-EN-NEW NLP-MBDS-EN
    os.chdir("NLP-MBDS-EN/07_rag")
print("Setup complete.")

If Colab warns that an already-imported package changed during the install, restart the runtime (Runtime, Restart session) and run the setup cell again. The second pass is instant.

## Step 1: A toy corpus and a preprocessing pipeline

Twelve tiny documents about skies, foxes, and breakfasts. Small on purpose: with a dozen documents we can read every ranking by eye and catch the engines when they get it wrong. The preprocessing is the classic sparse-retrieval recipe: lowercase, strip punctuation, tokenize, drop stopwords.


In [ ]:
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# A small mixed knowledge base, engineered so the three engines visibly disagree.
# Several documents answer a question using DIFFERENT words than the question would
# (synonyms, paraphrase); a few are "keyword traps" that share a word with a query
# but mean something else entirely.
documents = [
    "Refunds are issued within five business days of an approved request.",     # 0  refund
    "If you are unhappy with a purchase, we will reimburse you in full.",        # 1  reimburse
    "Tap the back arrow to return to the previous screen.",                      # 2  trap: 'back'
    "Counting your money by hand is slower than using a calculator.",            # 3  trap: 'money'
    "The cheetah is the fastest animal on land, reaching 100 km/h.",             # 4  fast animal
    "Lightning-quick reflexes help predators catch their prey.",                 # 5  speed (paraphrase)
    "Fast food is cheap, quick to order, and popular worldwide.",                # 6  trap: 'fast'
    "Reset your password using the link in the verification email.",             # 7
    "Lions and tigers are large members of the cat family.",                     # 8  felines
    "Paris is the capital of France and a major tourist city.",                  # 9
    "Our office is open from nine to five, Monday through Friday.",              # 10
    "A balanced diet includes fruit, vegetables, and whole grains.",             # 11
]

def preprocess(text):
    # lowercase + strip punctuation, then drop English stopwords
    text = ''.join(c.lower() for c in text if c not in string.punctuation)
    stop_words = set(stopwords.words('english'))
    return ' '.join(t for t in word_tokenize(text) if t not in stop_words)

preprocessed_docs = [preprocess(doc) for doc in documents]
print("before:", documents[0])
print("after: ", preprocessed_docs[0])

## Step 2: TF-IDF, the classic vector space model

Each document becomes a vector with one dimension per vocabulary term, weighted by TF-IDF. A weight is high when the term is frequent in this document but rare across the collection. The query becomes a vector in the same space, and relevance is the cosine of the angle between them. This is 1970s technology that still runs inside search engines today.


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fit the TF-IDF vocabulary on the preprocessed documents and weight every term.
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preprocessed_docs)
print(f"document-term matrix: {tfidf_matrix.shape[0]} docs x {tfidf_matrix.shape[1]} terms")

def tfidf_search(query, top_n=3):
    query_vector = vectorizer.transform([preprocess(query)])         # query into the same vector space
    sims = cosine_similarity(query_vector, tfidf_matrix).flatten()   # cosine of the angle to each document
    top = np.argsort(sims)[::-1][:top_n]                             # highest scores first
    return top.tolist(), sims[top].tolist()

for doc_id, score in zip(*tfidf_search("How do I get my money back?")):
    print(f"[{score:.2f}] {documents[doc_id]}")

## Step 3: BM25, TF-IDF with two screws to turn

BM25 is what actually ships inside Lucene, Elasticsearch, and OpenSearch. It fixes two blind spots of raw TF-IDF with two parameters. `k1` saturates term frequency (the tenth occurrence of a word should matter less than the second), and `b` normalizes by document length (a document should not win just by being long).


In [ ]:
from rank_bm25 import BM25Okapi

K1 = 1.5    # try: 0  (term-frequency saturation maxed out: repeated terms stop mattering at all)
B  = 0.75   # try: 0  (long documents are no longer penalized)

bm25 = BM25Okapi([doc.split() for doc in preprocessed_docs], k1=K1, b=B)

def bm25_search(query, top_n=3):
    scores = bm25.get_scores(preprocess(query).split())
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]
    return top, [scores[i] for i in top]

for doc_id, score in zip(*bm25_search("How do I get my money back?")):
    print(f"[{score:.2f}] {documents[doc_id]}")

## Step 4: Dense retrieval, searching by meaning

Both engines so far share one assumption: a document is relevant only if it contains the query's strings. Dense retrieval drops that assumption. A sentence-embedding model maps texts to vectors where geometric closeness means semantic closeness, so "azure heavens" can land next to "blue sky" without sharing a single character. Note that we feed it the raw documents, because embedding models want natural language, not stopword-stripped token soup.


In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
# try: "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", then query in Spanish

st_model = SentenceTransformer(EMBEDDING_MODEL)   # instant, the setup cell already cached it
document_embeddings = st_model.encode(documents, convert_to_tensor=True)
print(f"{document_embeddings.shape[0]} documents x {document_embeddings.shape[1]} dimensions")

def dense_search(query, top_n=3):
    query_embedding = st_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, document_embeddings)[0].cpu()
    top = torch.topk(scores, k=top_n)
    return top.indices.tolist(), top.values.tolist()

for doc_id, score in zip(*dense_search("How do I get my money back?")):
    print(f"[{score:.2f}] {documents[doc_id]}")

## Step 5: The face-off, three engines and one query

All three engines are now live on the same corpus. Run the default query, "How do I get my money back?". No document contains the words "money" or "back" in a way that answers it; the real answers talk about refunds and reimbursement. Watch which documents each engine ranks first.

The `# try` queries push it further: a request written entirely in synonyms (where the sparse engines score zero), and an exact rare term (where they are unbeatable). The second cell then drops onto the corpus we actually build on, the Helcon help center, and simply runs dense search, the engine the next session uses.


In [ ]:
QUERY = "How do I get my money back?"
# try: "a speedy creature"   (pure synonyms: the sparse engines score zero, dense finds the cheetah)
# try: "tell me about big cats"   (dense knows lions and tigers are felines; sparse shares no word)
# try: "cheetah"   (an exact rare term: BM25 is unbeatable here, dense can only tie it)
TOP_N = 3

results = {
    "TF-IDF": tfidf_search(QUERY, TOP_N),
    "BM25":   bm25_search(QUERY, TOP_N),
    "dense":  dense_search(QUERY, TOP_N),
}

print(f"QUERY: {QUERY!r}\n")
print(f"{'rank':<6}" + "".join(f"{name:<40}" for name in results))
print("-" * (6 + 40 * len(results)))
for r in range(TOP_N):
    row_cells = [f"[{scores[r]:5.2f}] {documents[ids[r]][:28]}" for ids, scores in results.values()]
    print(f"{r + 1:<6}" + "".join(f"{c:<40}" for c in row_cells))

In [ ]:
# Now the corpus the course is actually built on: the Helcon help center. This time we
# just run DENSE retrieval, the engine the next session uses. On a help center the three
# engines barely disagree: a support article and the questions people ask about it share
# so much vocabulary that even keyword matching does fine. The dramatic differences live
# on harder corpora like the toy one above; here we simply show semantic search answering
# a real question over our own documents.
import os, sys
for candidate in ("../system", "../../system", "system", "NLP-MBDS-EN/system"):
    if os.path.exists(os.path.join(candidate, "system_io.py")):
        sys.path.insert(0, candidate)
        break
from system_io import load_seed

helcon = load_seed("corpus")                       # KB articles + policies (our running system)
helcon_texts = [f"{d['title']}. {d['text']}" for d in helcon]
helcon_titles = [d["title"] for d in helcon]
helcon_emb = st_model.encode(helcon_texts, convert_to_tensor=True)
print(f"Helcon help center: {len(helcon)} documents (KB + policies)\n")

def helcon_dense_search(query, top_n=3):
    scores = util.cos_sim(st_model.encode(query, convert_to_tensor=True), helcon_emb)[0].cpu()
    top = torch.topk(scores, k=top_n)
    return top.indices.tolist(), top.values.tolist()

HELCON_QUERY = "How long do I have to ask for a refund?"
# try: "connect my Shopify store" / "I want to stop paying" / "who can see my data?"
print(f"QUERY: {HELCON_QUERY!r}\n")
for doc_id, score in zip(*helcon_dense_search(HELCON_QUERY)):
    print(f"[{score:.2f}] {helcon_titles[doc_id]}")

## Takeaway

The default query never says "refund" or "reimburse", yet those are the only documents that answer it. TF-IDF and BM25 rank the keyword traps instead, the document that happens to contain "money" and the one that contains "back", because string matching is all they can do. They retrieve on shared words, not on meaning. The dense engine ranks the real refund documents first, because it matched the question. The synonym queries make the gap total: "a speedy creature" shares no word with any document, so both sparse engines score zero while dense still finds the cheetah. The one place sparse wins is the exact rare term "cheetah", where BM25's idf weighting is unbeatable and embeddings can only tie it. Neither family dominates everywhere, which is why production systems run both and merge the results. That is hybrid retrieval, the subject of the final project.
